# 5. Classify, shortlist and score

Load the best classifier and best high-popularity regressor. Both receive original features and apply their own saved preprocessing. This replaces the original missing-model path and incompatible PCA hand-off. The classifier is reused instead of running another 1,500-candidate search.

In [1]:
from pipeline_lib import RunContext, run_combined, write_report
ctx = RunContext.from_environment("combined")
print("Profile:", ctx.config["profile"])
print("Artifacts:", ctx.run_dir)
from IPython.display import Markdown, display

Profile: smoke
Artifacts: /Users/anishech/Projects/ML_Spotify/runs/20261021_135938_718078_smoke


Use the shared holdout only. Keep tracks predicted as hits, score their popularity and sort by predicted score. Track names and artists are joined back for interpretation. Shortlist regression includes false positives; it does not measure performance on all songs.

In [2]:
ranked_candidates = run_combined(ctx)
display(ranked_candidates)

Combined holdout: 201 shortlisted / 1600; metrics {'rows': 1600, 'actual_hits': 51, 'predicted_hits': 201, 'accuracy': 0.89375, 'balanced_accuracy': 0.8503145609438094, 'hit_precision': 0.20398009950248755, 'hit_recall': 0.803921568627451, 'hit_f1': 0.3253968253968254, 'confusion_matrix': [[1389, 160], [10, 41]], 'average_precision': 0.2482557858716623, 'roc_auc': 0.9001126596539197, 'shortlist_regression': {'rows': 201, 'mae': 28.148268362468336, 'rmse': 38.4921767675666, 'r2': -1.0831664971697457, 'within_5_percent': 17.91044776119403}, 'shortlist_mean_baseline': {'rows': 201, 'mae': 28.4354091610911, 'rmse': 38.50364369088886, 'r2': -1.0844078438834805, 'within_5_percent': 18.407960199004975}}


,row_id,actual_popularity,predicted_popularity,hit_score,track_id,artists,album_name,track_name
0,68978,38,78.760971,0.863203,75Bk2jHaVd5kR5KDDRsnT8,Franky Style,EXPLÍCITO,RUNAWAY
1,81074,3,78.748611,0.738238,4g7118rCobKcU4XQXSFlBS,The Weeknd,Drippy Drippy,The Hills
2,68935,41,78.577481,0.877194,6JXbjTYnHhVpgz8aRm1wTR,Balbi El Chamako,La Vida Es Bela,Ando Con el Carro
3,68873,41,78.104307,0.852591,2L5aP9ytaSmHqAAuaK4Vpk,Young Cister,Xtaciss,Lemon Kush
4,68941,39,77.935561,0.650006,3JCcBtwNldEBd3O4hqYDWI,Almighty;Manny Montes,Genelipsis,No Me Arrodillo
5,56058,83,77.851892,0.659818,1ZdhOMWyFR8Iv9eylMGYg2,Labrinth,Imagination & the Misfit Kid,Mount Everest
6,68602,87,77.821162,0.748530,2JPLbjOn0wPCngEot2STUS,Bad Bunny,Yonaguni,Yonaguni
7,67818,2,77.796061,0.773658,0ZkZtswuplfbjbXk7Y0jOv,Bad Gyal,Frescura y Perreo,Hookah
8,20409,75,77.654363,0.781396,09AvsbivV1FQBRAEBJh7UL,The Chainsmokers,So Far So Good,High
9,47770,62,77.635324,0.588456,38fIaph07Kd8ZIN6l17ZJs,Metallica,Kill 'Em All (Remastered),Whiplash - Remastered


In [3]:
report = write_report(ctx)
display(Markdown(report.read_text()))

# Spotify results: smoke

This run predicts Spotify popularity, not measured Billboard chart membership.

Raw rows: 114,000; cleaned: 71,945; modeled: 8,000.
Training rows: 6,400; holdout rows: 1,600 (51 actual hits). Hit means popularity > 70.

All stages share a saved stratified holdout. Every estimator stores its own genre encoder, scaler, PCA and model. Preprocessing is fitted within training folds; genre training encodings are cross-fitted.
Winners are selected using cross-validation, never holdout scores. Training metrics use the final refitted encoder and are descriptive only.

## Measured results

### classification: RandomForestClassifier

Accuracy 89.38%; majority baseline 96.81%.
Hit precision 20.40%; recall 80.39%; F1 0.3254; average precision 0.2483.
Confusion matrix [[TN, FP], [FN, TP]]: [[1389, 160], [10, 41]].
Selection metric: hit_f1; CV score 0.2635.

### regression: Ridge

MAE 11.188; RMSE 15.510; R² 0.361; within ±5 points 36.00%.
Mean-prediction baseline: MAE 16.179; RMSE 19.415; R² -0.001.
Selection metric: neg_rmse; CV score -15.6774.

### regression_high: ElasticNet

MAE 3.018; RMSE 3.791; R² 0.141; within ±5 points 84.31%.
Mean-prediction baseline: MAE 3.286; RMSE 4.093; R² -0.001.
Selection metric: neg_rmse; CV score -4.5789.

## Combined workflow

Shortlisted 201 tracks from 1600 holdout tracks.
The high-popularity regressor scores all shortlisted tracks, including false positives. These metrics describe the shortlist, not every track or future Billboard success.
Shortlist regression: {"rows": 201, "mae": 28.148268362468336, "rmse": 38.4921767675666, "r2": -1.0831664971697457, "within_5_percent": 17.91044776119403}.

## Interpretation and limits

A majority-class baseline can have high accuracy while finding zero hits. Use hit precision, recall, F1 and average precision to judge discovery. Regression R² must be compared to a mean baseline. Within ±5 points is a tolerance metric, not classification accuracy.

The original saved RF classifier had 96.70% accuracy on a different random split. The original general-regression Lasso had RMSE 15.542 and R² 0.371; the high-popularity Lasso had RMSE 4.179 and R² 0.223. These are historical notebook outputs, not rerun results. Different samples, stratification, cross-fitted encoding, valid validation and selection criteria mean this run is not a direct reproduction of those numbers.

Random track splitting does not test future releases or independence between artists. Song characteristics alone omit marketing, audience and release timing. The high-popularity model is trained on known hits and may overestimate false positives.

This is a small-sample pipeline verification. Its model rankings and metrics are unstable; use the full-data run for the next experiment.

Open `metrics/*_summary.csv` for all models, `predictions/ranked_candidates.csv` for the shortlist, and `executed_notebooks/` for cell-by-cell outputs. Timing and hashes are recorded in `run_manifest.json`.


Artifacts include per-model metrics, CV tables, predictions, complete model pipelines, plots and executed notebooks. The run manifest records configuration, timing, interpreter, input and source hashes. Read `RESULTS.md` for this run; a smoke test is a workflow check, not the final model comparison.